# ViFinQA P2.2 — Structured Selection v2

**Settings:** GPU T4 x2, Internet On, attach exactly one `vifinqa-payload` dataset. Payload must be schema 8. Run only stage **B-semantic-v5 first**; run C only after downloading and auditing B locally.

In [ ]:
import glob, hashlib, json, pathlib
hits = glob.glob('/kaggle/input/**/retrieval.jsonl', recursive=True)
assert len(hits) == 1, f'Attach exactly one payload dataset: {hits}'
PAYLOAD = str(pathlib.Path(hits[0]).parent)
manifest_path = pathlib.Path(PAYLOAD) / 'payload-manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest.get('schema_version') == 8, manifest.get('schema_version')
assert manifest.get('fuzzy_scorer') == {'backend': 'difflib.SequenceMatcher', 'version': '1'}
for rel, count in [('targets/p22b_semantic_groundable_v5.json', 2), ('targets/p22c_semantic_groundable_v5.json', 4)]:
    path = pathlib.Path(PAYLOAD) / rel
    obj = json.loads(path.read_text(encoding='utf-8'))
    assert obj['count'] == count == len(obj['ids']) == len(set(obj['ids']))
    assert rel in manifest['files']
print('PAYLOAD', PAYLOAD, '| schema', manifest['schema_version'], '| files', len(manifest['files']))
import torch
print('GPUs', torch.cuda.device_count(), torch.cuda.get_device_name(0))

In [ ]:
import pathlib, shutil
SRC = pathlib.Path(PAYLOAD) / 'code'
DST = pathlib.Path('/kaggle/working/code')
shutil.rmtree(DST, ignore_errors=True)
shutil.copytree(SRC, DST)
print('code ->', DST)

In [ ]:
%%time
!pip install -q "transformers>=4.45,<5" "accelerate>=1,<2" "bitsandbytes>=0.45,<1"
import transformers, bitsandbytes
print('transformers', transformers.__version__, 'bitsandbytes', bitsandbytes.__version__)

## Smoke runtime

Smoke uses a separate output, no mask and `target=all` so it definitely exercises v2. Never resume it into B/C.

In [ ]:
%%time
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-14B-Instruct --load-4bit \
    --llm-mode select_v2 --llm-target all \
    --out /kaggle/working/codegen_p22_smoke.jsonl --limit 12 --no-resume \
    --n 1 --temperature 0.2 --k 0 --max-tokens 512 --max-input-tokens 6000 \
    --batch-size 1 --checkpoint-every 4 --time-budget-min 40 --seed 13

In [ ]:
import collections, json, math, pathlib
smoke = [json.loads(x) for x in pathlib.Path('/kaggle/working/codegen_p22_smoke.jsonl').open(encoding='utf-8')]
print('rows', len(smoke), 'sources', collections.Counter(r['source'] for r in smoke))
print('outcomes', collections.Counter((r.get('selection_trace') or {}).get('outcome', 'not_run') for r in smoke))
assert len(smoke) == 12 and all(math.isfinite(float(r['answer'])) for r in smoke)

## Stage B-semantic-v5 — 2 fully grounded shortlists

No rescue in B. Each F-slot passed planner completeness plus metric, entity, period and exact-year grounding. Output remains a complete 1,012-row checkpoint.

In [ ]:
%%time
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-14B-Instruct --load-4bit \
    --llm-mode select_v2 --llm-target empty \
    --llm-ids-file targets/p22b_semantic_groundable_v5.json \
    --out /kaggle/working/codegen_p22b_semantic_v5_sel14b.jsonl \
    --n 2 --temperature 0.2 --k 0 --max-tokens 512 --max-input-tokens 6000 \
    --batch-size 1 --checkpoint-every 1 --time-budget-min 120 --seed 13

In [ ]:
def qa_masked(path, mask_rel):
    rows = [json.loads(x) for x in pathlib.Path(path).open(encoding='utf-8')]
    mask = set(json.loads((pathlib.Path(PAYLOAD) / mask_rel).read_text(encoding='utf-8'))['ids'])
    ids = [int(r['id']) for r in rows]
    attempted = {int(r['id']) for r in rows if r.get('llm_attempt_status') == 'completed'}
    assert len(rows) == 1012 and len(ids) == len(set(ids))
    assert all(math.isfinite(float(r['answer'])) for r in rows)
    assert len({r.get('run_signature') for r in rows}) == 1
    assert attempted <= mask, sorted(attempted - mask)[:10]
    pending = mask - attempted
    outcomes = collections.Counter((r.get('selection_trace') or {}).get('outcome') for r in rows if int(r['id']) in attempted)
    print('target', len(mask), 'attempted', len(attempted), 'pending', len(pending), 'outcomes', outcomes)
    return pending
pending_b = qa_masked('/kaggle/working/codegen_p22b_semantic_v5_sel14b.jsonl', 'targets/p22b_semantic_groundable_v5.json')
assert not pending_b, 'Resume the exact Stage B command before download'

## Stage C-semantic-v5 — 4 fully grounded rescue questions

STOP after B. Run C only after B has been downloaded and passed the local audit/replay gates. C has a disjoint mask, enables rescue and uses a separate output/signature.

In [ ]:
APPROVE_STAGE_C = False  # set True only after local audit of B
assert APPROVE_STAGE_C, 'STOP: download and audit B locally before Stage C'
!python /kaggle/working/code/kaggle_codegen.py --payload $PAYLOAD --backend hf \
    --model Qwen/Qwen2.5-Coder-14B-Instruct --load-4bit \
    --llm-mode select_v2 --llm-target empty \
    --llm-ids-file targets/p22c_semantic_groundable_v5.json \
    --out /kaggle/working/codegen_p22c_semantic_v5_sel14b.jsonl \
    --n 2 --temperature 0.2 --k 0 --rescue-no-candidates \
    --rescue-table-k 20 --rescue-min-score 28 \
    --max-tokens 512 --max-input-tokens 6000 --batch-size 1 \
    --checkpoint-every 1 --time-budget-min 180 --seed 13

In [ ]:
pending_c = qa_masked('/kaggle/working/codegen_p22c_semantic_v5_sel14b.jsonl', 'targets/p22c_semantic_groundable_v5.json')
assert not pending_c, 'Resume the exact Stage C command before download'

## Download and local merge

Download `codegen_p22b_semantic_v5_sel14b.jsonl` first. Audit and CPU-replay B into frozen #19. If B is sound, return to run C; download/audit C and replay it into the B hybrid. Exact commands are in `RUNBOOK_P2_2_STRUCTURED_SELECTION_V2.md`.

Resume only with the same output and every flag unchanged. Changing model revision, package versions, batch size, checkpoint size, token limits, mask, rescue flags or temperature creates a different run contract; start a new output file.